In [ ]:
import h5py
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

In [ ]:
raw_dir = Path("/path/to/file/").expanduser()
files = [f"file_{i:05d}_{i * 60:07_d}.h5" for i in range(1, 1009)]
raw_file = raw_dir / files[0]

In [ ]:
with h5py.File(raw_file, "r") as f:
    data16 = f["MeasurementData/Channels/0/DataUint16"][:]
    data8 = f["MeasurementData/Channels/0/DataUint8"][:]

i = data16[:, 0]
x = data16[:, 1]
y = data16[:, 2]
b = data8[:, 0]

bitmap = np.zeros((y.max() + 1, x.max() + 1))
bitmap[y, x] = b
bitmap = (bitmap > 0).astype(np.uint8) * 255

Image.fromarray(bitmap, mode="L").save("bitmap.png")

In [ ]:
img = cv2.imread("bitmap.png")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = cv2.medianBlur(gray, 5)
circles = cv2.HoughCircles(
    gray,
    cv2.HOUGH_GRADIENT,
    dp=1,
    minDist=100,      
    param1=100,         
    param2=10,       
    minRadius=70,       
    maxRadius=80
)
if circles is not None:
    circles = np.uint16(np.around(circles))[0]
    for x, y, r in circles:
        cv2.circle(img, (x, y), r, (0, 255, 0), 2)
        cv2.putText(img, f"({x}, {y}, {r})", (x - r, y), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 1)
plt.figure(figsize=(32, 32))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
c_x = [496, 866, 1236, 1608, 1978]
c_y = [518, 888, 1258, 1630, 2000]
r = 75
p = [
    "01.06", "02.01", "03.09", "04.24", "05.20", "06.07", "07.19", "08.16", "09.12", "10.05", "11.15", "12.10", "13.11",
    "14.18", "15.17", "16.13", "17.08", "18.22", "19.02", "20.21", "21.23", "22.14", "23.25", "24.04", "25.03"
][::-1]
with open("coordinates.csv", "w") as f:
    f.write("x,y,part")
    for y in c_y[::-1]:
        for x in c_x:
            f.write(f"\n{x},{y},{p.pop()}")